In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, date

In [0]:
landing = 'workspace.google_drive'

In [0]:
%sql
show schemas in workspace.google_drive

In [0]:
%sql
select * from workspace.google_drive.batting_batsman_data_1_st_match_n_ 

In [0]:
landing_path = '/Volumes/ipl_database/default/ipl_landing_volume'

In [0]:
spark.table('workspace.google_drive.batting_batsman_data_4_th_match_d_n_').filter(col('_fivetran_synced') < max_updated_at).display()

In [0]:
%sql
select to_timestamp(`_fivetran_synced`) from workspace.google_drive.batting_batsman_data_4_th_match_d_n_

In [0]:
%sql
-- get all tables unser schema
show tables in workspace.google_drive

In [0]:
batting_data = spark.sql('''
                         show 
                         tables 
                         in workspace.google_drive
                         ''').filter("tableName like 'batting%'").collect()
batting_files = [row.tableName for row in batting_data]

In [0]:
batting_files

In [0]:
type(max_updated_at)

In [0]:
batsman_raw_df = spark.table('ipl_database.bronze.batsman_raw')
max_updated_at = batsman_raw_df.select(max('updated_at')).collect()[0][0]
print(max_updated_at)

In [0]:
%sql
drop table ipl_database.bronze.batsman_raw

In [0]:
try:
    batsman_raw_df = spark.table('ipl_database.bronze.batsman_raw')
    max_updated_at = batsman_raw_df.select(max('updated_at')).collect()[0][0]
    for row in batting_data:
        batting_df = spark.table(f'workspace.google_drive.{row.tableName}').filter(col("_fivetran_synced") > max_updated_at)
        if not batting_df.isEmpty:
            batting_df = batting_df.withColumn(
                "updated_at",
                current_timestamp()
            )
            batting_df.createOrReplaceTempView("batting_df")
            spark.sql('''
                merge into ipl_database.bronze.batsman_raw t
                using batting_df b 
                on t.id = b.id and t.date = b.date
                when not matched then insert *
            ''')
            print(f"File {row.tableName} appended to batsman.raw")
        else:
            print("New data not present")
except Exception as e:
    try:
        for row in batting_data:
            batting_df = spark.table(f'workspace.google_drive.{row.tableName}')
            batting_df = batting_df.withColumn(
                "updated_at",
                current_timestamp()
            )
            batting_df.write.format('delta').mode("append").option("mergeSchema", "true").saveAsTable("ipl_database.bronze.batsman_raw")
    except Exception:
        print("Failed to load the data.")
    print(f"raw data ingestion failed due to {e}")

In [0]:
spark.table('ipl_database.bronze.batsman_raw').filter(col('_fivetran_synced') < max_updated_at).display()